# KUM值与优化后授信额度：图A、图B

本 Notebook 只调用已保存的全量额度优化结果，不重新训练或优化。

- **A：优化后授信额度**
- **B：KUM值**
- 等级顺序：`F3 → F2 → F1 → E → D → C → B → A`
- **图A**：A、B 全局 z-score 标准化后，各等级均值及 95% 置信区间。
- **图B**：各等级原始均值点 `(平均优化额度, 平均KUM值)` 的散点轨迹。

文件名或字段名不一致时，只修改下一格。

In [ ]:
# ======================== 可修改配置区 ========================
RESULT_FILE = 'reports/full_crossfit_optimization/credit_limit_large_grid_results.csv'
CLEANED_FILE = 'data_cleaned.csv'
OUTPUT_DIR = 'reports/full_crossfit_optimization/viz_kum_AB'
CSV_ENCODING = 'utf-8-sig'

RESULT_ID_COL = None          # 默认自动识别 customer_id / cst_id
CLEANED_ID_COL = None         # 默认自动识别 cst_id / customer_id
KUM_COL = None                # 预处理后的正式名称为“kum分”
OPTIMIZED_LIMIT_COL = None    # 默认识别 credit_limit
TALENT_LEVEL_COL = None       # 默认识别 talent_level

CI_Z = 1.96                   # 正态近似双侧95%置信区间
FIG_DPI = 180
# =============================================================

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

TIER_ORDER = [1, 2, 3, 4, 5, 6, 7, 8]
TIER_LABEL = {1: 'F3', 2: 'F2', 3: 'F1', 4: 'E', 5: 'D', 6: 'C', 7: 'B', 8: 'A'}
TIER_TEXT_TO_NUM = {'F3': 1, 'F2': 2, 'F1': 3, 'E': 4, 'D': 5, 'C': 6, 'B': 7, 'A': 8}
money_wan = FuncFormatter(lambda x, _: f'{x / 10000:,.0f}万')

def resolve_file(configured_path, exact_name):
    path = Path(configured_path)
    if path.is_file():
        return path
    matches = sorted(Path('.').rglob(exact_name))
    if len(matches) == 1:
        print(f'配置路径不存在，已自动找到: {matches[0]}')
        return matches[0]
    if not matches:
        raise FileNotFoundError(f'未找到 {configured_path}，请确认原优化结果已保存，或修改配置区路径。')
    raise FileNotFoundError(f'找到多个 {exact_name}: {matches}，请在配置区指定正确路径。')

def read_table(path):
    path = Path(path)
    if path.suffix.lower() == '.csv':
        try:
            return pd.read_csv(path, encoding=CSV_ENCODING, dtype=str)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding='gbk', dtype=str)
    if path.suffix.lower() in ('.xlsx', '.xls'):
        return pd.read_excel(path, dtype=str)
    raise ValueError(f'仅支持CSV/Excel文件: {path}')

def choose_col(df, configured, candidates, label):
    if configured is not None:
        if configured not in df.columns:
            raise KeyError(f'{label}字段 {configured!r} 不存在。现有字段: {list(df.columns)}')
        return configured
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    raise KeyError(f'无法识别{label}。候选: {candidates}；现有字段: {list(df.columns)}')

def normalize_talent_level(series):
    raw = series.astype('string').str.strip()
    numeric = pd.to_numeric(raw, errors='coerce')
    text = raw.str.upper().str.replace('级', '', regex=False).map(TIER_TEXT_TO_NUM)
    return numeric.fillna(text)

In [ ]:
# 读取、识别字段并按客户ID一对一合并
result_path = resolve_file(RESULT_FILE, 'credit_limit_large_grid_results.csv')
cleaned_path = resolve_file(CLEANED_FILE, Path(CLEANED_FILE).name)
results = read_table(result_path)
cleaned = read_table(cleaned_path)

result_id = choose_col(results, RESULT_ID_COL, ['customer_id', 'cst_id', 'cust_id'], '优化结果客户ID')
cleaned_id = choose_col(cleaned, CLEANED_ID_COL, ['cst_id', 'customer_id', 'cust_id'], '清洗数据客户ID')
kum_col = choose_col(cleaned, KUM_COL, ['kum分', 'kum_score', 'KUM分', 'KUM_SCORE', 'kum', 'KUM'], 'KUM值')
limit_col = choose_col(results, OPTIMIZED_LIMIT_COL, ['credit_limit', 'optimized_credit_limit', 'optimal_credit_limit'], '优化后额度')
tier_col = choose_col(results, TALENT_LEVEL_COL, ['talent_level', 'talent_tier', 'tier', '档位'], '人才等级')

left = results[[result_id, limit_col, tier_col]].copy()
right = cleaned[[cleaned_id, kum_col]].copy()
left.columns = ['customer_id', 'optimized_credit_limit', 'talent_level']
right.columns = ['customer_id', 'kum_score']
left['customer_id'] = left['customer_id'].astype('string').str.strip()
right['customer_id'] = right['customer_id'].astype('string').str.strip()
if left['customer_id'].duplicated().any():
    raise ValueError('优化结果客户ID存在重复，无法一对一合并。')
if right['customer_id'].duplicated().any():
    raise ValueError('清洗数据客户ID存在重复，无法一对一合并。')

data = left.merge(right, on='customer_id', how='left', validate='one_to_one', indicator=True)
unmatched = int(data['_merge'].ne('both').sum())
data = data.drop(columns='_merge')
data['optimized_credit_limit'] = pd.to_numeric(data['optimized_credit_limit'], errors='coerce')
data['kum_score'] = pd.to_numeric(data['kum_score'], errors='coerce')
data['talent_level'] = normalize_talent_level(data['talent_level'])
invalid = data[['optimized_credit_limit', 'kum_score', 'talent_level']].isna().any(axis=1)
if invalid.any():
    raise ValueError(f'存在 {int(invalid.sum()):,} 行无有效KUM/额度/等级，其中ID未匹配 {unmatched:,} 行。')
data['talent_level'] = data['talent_level'].astype(int)
unexpected = sorted(set(data['talent_level']) - set(TIER_ORDER))
if unexpected:
    raise ValueError(f'发现预期1–8之外的人才等级: {unexpected}')
data['talent_label'] = data['talent_level'].map(TIER_LABEL)

# A、B均使用全体客户的均值和样本标准差进行全局标准化。
for raw_col, z_col in [('optimized_credit_limit', 'A_z'), ('kum_score', 'B_z')]:
    sd = data[raw_col].std(ddof=1)
    if not np.isfinite(sd) or sd == 0:
        raise ValueError(f'{raw_col} 的标准差为0或无效，无法标准化。')
    data[z_col] = (data[raw_col] - data[raw_col].mean()) / sd

present_levels = [level for level in TIER_ORDER if level in set(data['talent_level'])]
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
print(f'优化结果: {result_path} ({len(results):,}行)')
print(f'清洗数据: {cleaned_path} ({len(cleaned):,}行)')
print(f'字段: KUM={kum_col}, 优化额度={limit_col}, 等级={tier_col}')
print(f'成功对齐 {len(data):,} 位客户；等级: {[TIER_LABEL[x] for x in present_levels]}')
data.head()

## 图A：标准化均值 ± 95% CI 随等级变化

置信区间按每个等级内标准化变量的 `均值 ± 1.96 × 标准误` 计算。标准化参数来自全部有效客户，而不是各等级内部。

In [ ]:
summary = (data.groupby('talent_level')
           .agg(n=('customer_id', 'size'),
                A_raw_mean=('optimized_credit_limit', 'mean'),
                B_raw_mean=('kum_score', 'mean'),
                A_z_mean=('A_z', 'mean'), A_z_sd=('A_z', 'std'),
                B_z_mean=('B_z', 'mean'), B_z_sd=('B_z', 'std'))
           .reindex(present_levels).reset_index())
summary['talent_label'] = summary['talent_level'].map(TIER_LABEL)
summary['A_ci95'] = CI_Z * summary['A_z_sd'].fillna(0) / np.sqrt(summary['n'])
summary['B_ci95'] = CI_Z * summary['B_z_sd'].fillna(0) / np.sqrt(summary['n'])
x = np.arange(len(summary))

fig, ax = plt.subplots(figsize=(12, 6.8))
ax.errorbar(x, summary['A_z_mean'], yerr=summary['A_ci95'], color='#4E79A7',
            marker='o', markersize=7, linewidth=2.3, capsize=4, label='A：优化后授信额度')
ax.errorbar(x, summary['B_z_mean'], yerr=summary['B_ci95'], color='#F28E2B',
            marker='s', markersize=6.5, linewidth=2.3, capsize=4, label='B：KUM值')
ax.axhline(0, color='#777777', linewidth=1, linestyle='--', alpha=.65)
ax.set_xticks(x, summary['talent_label'])
ax.set_xlabel('人才等级（由低到高）')
ax.set_ylabel('全局标准化均值（z-score）')
ax.set_title('图A  优化后授信额度与KUM值的等级均值轨迹（95% CI）')
ax.grid(axis='y', linestyle='--', alpha=.25)
ax.legend(frameon=False, ncol=2, loc='upper left')
fig.tight_layout()
fig_a_path = out_dir / 'figure_A_standardized_means_95CI_by_talent.png'
fig.savefig(fig_a_path, dpi=FIG_DPI, bbox_inches='tight')
plt.show()
summary.to_csv(out_dir / 'talent_level_mean_ci_summary.csv', index=False, encoding='utf-8-sig')
print('图A已保存:', fig_a_path)
summary

## 图B：各等级 `(平均优化额度, 平均KUM值)` 散点轨迹

每个等级只对应一个均值点，并按 `F3 → … → A` 连接；点旁标注等级。横纵轴保留原始业务量纲。

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 7.5))
colors = plt.cm.viridis(np.linspace(.12, .9, len(summary)))
ax.plot(summary['A_raw_mean'], summary['B_raw_mean'], color='#808080', linewidth=1.8, alpha=.8, zorder=1)
ax.scatter(summary['A_raw_mean'], summary['B_raw_mean'], s=115, c=colors,
           edgecolor='white', linewidth=1.2, zorder=2)
for i, row in summary.iterrows():
    ax.annotate(row['talent_label'], (row['A_raw_mean'], row['B_raw_mean']),
                xytext=(7, 7), textcoords='offset points', fontsize=11, fontweight='bold')
# 用小箭头强调等级提升方向。
for i in range(len(summary) - 1):
    ax.annotate('', xy=(summary.loc[i + 1, 'A_raw_mean'], summary.loc[i + 1, 'B_raw_mean']),
                xytext=(summary.loc[i, 'A_raw_mean'], summary.loc[i, 'B_raw_mean']),
                arrowprops=dict(arrowstyle='->', color='#808080', lw=1.2, shrinkA=10, shrinkB=10))
ax.set_xlabel('A：各等级平均优化后授信额度（元）')
ax.set_ylabel('B：各等级平均KUM值')
ax.xaxis.set_major_formatter(money_wan)
ax.set_title('图B  优化后授信额度与KUM值的等级均值散点轨迹')
ax.grid(linestyle='--', alpha=.25)
fig.tight_layout()
fig_b_path = out_dir / 'figure_B_level_mean_scatter_trajectory.png'
fig.savefig(fig_b_path, dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print('图B已保存:', fig_b_path)
summary[['talent_label', 'n', 'A_raw_mean', 'B_raw_mean']]

### 解读注意

图A用于比较两条等级梯度，图B用于展示两个等级均值的联合轨迹。95% CI 描述的是等级样本均值的不确定性；两图均为描述性关系，不代表因果效应。